In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error



In [ ]:
df = pd.read_csv('/home/insurance.csv')

In [ ]:
df.head()

,age,sex,bmi,children,smoker,region,expenses
0,19,female,27.9,0,yes,southwest,16884.92
1,18,male,33.8,1,no,southeast,1725.55
2,28,male,33.0,3,no,southeast,4449.46
3,33,male,22.7,0,no,northwest,21984.47
4,32,male,28.9,0,no,northwest,3866.86


In [ ]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
expenses,0


In [ ]:

# Step 1 -> train/test/split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['expenses']),
                                                 df['expenses'],
                                                 test_size=0.2,
                                                random_state=42)

In [ ]:
X_train.head(2)

,age,sex,bmi,children,smoker,region
560,46,female,20.0,2,no,northwest
1285,47,female,24.3,0,no,northeast


In [ ]:
y_train.head()

,expenses
560,9193.84
1285,8534.67
1142,27117.99
969,8596.83
486,12475.35


In [ ]:
# Ordinalencoding
oe = OrdinalEncoder(categories=[['no','yes']])
X_train_smoker = oe.fit_transform(X_train[['smoker']])
X_test_smoker = oe.fit_transform(X_test[['smoker']])
X_train_smoker.shape

(1070, 1)

In [ ]:
X_test_smoker.shape

####1.Before using pipeline

In [ ]:
# OneHotEncoding
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_sex_region = ohe.fit_transform(X_train[['sex','region']])
X_test_sex_region = ohe.fit_transform(X_test[['sex','region']])
X_train_sex_region.shape

(1070, 4)

In [ ]:
X_test_sex_region.shape

(268, 4)

In [ ]:
X_train_transform = np.hstack((
    X_train[['age']],
    X_train[['bmi']],
    X_train_sex_region,
    X_train[['children']],
    X_train_smoker
))
X_test_transform = np.hstack((
    X_test[['age']],
    X_test[['bmi']],
    X_test_sex_region,
    X_test[['children']],
    X_test_smoker
))
X_train_transform.shape

(1070, 8)

In [ ]:
clf = DecisionTreeRegressor()
clf.fit(X_train_transform, y_train)

DecisionTreeRegressor()

In [ ]:
y_pred = clf.predict(X_test_transform)
y_pred

array([ 8615.3 ,  4571.41, 28950.47,  9225.26, 33732.69, 11326.71,
        2709.24, 14410.93,  2974.13, 10493.95, 19362.  ,  8538.29,
        4040.56, 46200.99, 48970.25, 48885.14,  9304.7 , 41676.08,
        8232.64, 21348.71,  3906.13,  7740.34,  1639.56,  1984.45,
       10493.95, 11512.41, 13635.64,  4618.08,  9225.26,  1712.23,
        7512.27, 11840.78, 11482.63,  4906.41,  3490.55, 12797.21,
        2007.95,  6849.03, 24667.42, 37742.58,  4561.19,  2362.23,
       10713.64, 12323.94,  4949.76, 12913.99, 26018.95,  3906.13,
       40273.65, 24915.05, 13887.97,  1720.35,  6393.6 ,  1708.  ,
       20781.49, 10600.55,  3268.85, 58571.07, 11362.76, 11512.41,
       13393.76,  4686.39, 31620.  ,  8252.28, 10797.34,  4337.74,
       21472.48, 12479.71,  2842.76,  1984.45,  5488.26,  8334.46,
        8083.92,  7682.67,  7147.47,  5469.01,  4449.46, 10355.64,
        4058.71,  8615.3 ,  1261.44, 26109.33,  4934.71, 37484.45,
       42112.24, 41919.1 ,  4433.39, 11353.23,  7727.25, 11253

In [ ]:
from sklearn.metrics import mean_squared_error

mean_squared_error(y_test, y_pred)

43328668.80309105

####2.After Using Pipeline

In [ ]:
Transform1 = ColumnTransformer(transformers=[
    ('smoker_transform',OrdinalEncoder(categories=[['no','yes']]),['smoker'])

],remainder='passthrough')

In [ ]:
Transform2 = ColumnTransformer(transformers=[
    ('onehot_sex_region',OneHotEncoder(sparse_output=False,drop='first'),['sex','region'])
],remainder='passthrough')

In [ ]:
Transform3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,7))
])
Transform3

ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 7, None))])

In [ ]:
# Feature selection
Transform4 = SelectKBest(score_func=chi2,k=8)

In [ ]:
Transform5 = DecisionTreeRegressor()


In [ ]:
pipe = Pipeline([
    ('Transform1',Transform1),
    ('Transform2',Transform2),
    ('Transform3',Transform3),
    ('Transform4',Transform4),
    ('Transform5',Transform5)
])

In [ ]:
# Alternate Syntax
pipe = make_pipeline(Transform1,Transform2,Transform3,Transform4,Transform5)

In [ ]:
# Code here
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('smoker_transform',
                                  OrdinalEncoder(categories=[['no', 'yes']]),
                                  ['smoker'])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('onehot_sex_region',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  ['sex', 'region'])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 7, None))]),
 'selectkbest': SelectKBest(k=8, score_func=<function chi2 at 0x7ed612bd8c20>),
 'decisiontreeregressor': DecisionTreeRegressor()}